# Install Dependencies

* Purpose: Installs required Python packages

* Packages:

  * chromadb: Vector database for storage/retrieval

  * google-generativeai: Access to Gemini models

  * pandas: Data manipulation/analysis

  * tqdm: Progress tracking

* -q flag reduces installation output noise

In [ ]:
!pip install chromadb google-generativeai pandas tqdm -q

#  Import Libraries

* chromadb: Vector database operations

* genai: Gemini API access

* userdata: Secure access to Colab secrets

* pd: Dataframe handling

* tqdm: Visual progress bars

* List: Type hinting for function signatures

In [30]:
import chromadb
import google.generativeai as genai
from google.colab import userdata
import pandas as pd
from tqdm.notebook import tqdm
from typing import List

# Initialize Gemini API

* Retrieves Gemini API key from Colab secrets

* Configures Gemini client with the API key

* Essential for authenticating embedding requests

In [31]:
GEMINI_API_KEY = userdata.get('GEMINI_NEW')
genai.configure(api_key=GEMINI_API_KEY)

# Fixed Gemini Embedding Function

* Purpose: Custom embedding interface for ChromaDB

* Key Features:

  * Uses Gemini's embedding-001 model

  * Implements __call__ method required by ChromaDB

  * Handles batch embedding requests

  * RETRIEVAL_DOCUMENT optimizes for search tasks

  * name() provides unique identifier for ChromaDB

In [32]:
class GeminiEmbeddingFunction:
    def __init__(self, model_name="models/embedding-001"):
        self.model_name = model_name

    def __call__(self, input: List[str]) -> List[List[float]]:
        """Correct signature: takes list of strings, returns list of embeddings"""
        return genai.embed_content(
            model=self.model_name,
            content=input,
            task_type="RETRIEVAL_DOCUMENT"
        )['embedding']

    def name(self):
        return "gemini-embedding-function"

# Initialize ChromaDB

* Creates persistent ChromaDB client

* Stores data in /content/chroma_db directory

* Instantiates the custom Gemini embedding function

In [33]:
client = chromadb.PersistentClient(path="/content/chroma_db")
gemini_ef = GeminiEmbeddingFunction()

# Create or get collection

* Creates "tech_docs" collection (or loads existing)

* Links collection to Gemini embedding function

* All operations will use Gemini for embeddings

In [34]:
collection = client.get_or_create_collection(
    name="tech_docs",
    embedding_function=gemini_ef
)

# Add sample documents with metadata

* Data Preparation:

  * Sample documents about vector databases

  * Metadata for categorization

  * Unique IDs for each document

* upsert(): Inserts or updates documents in collection

* Documents are automatically embedded using Gemini

In [35]:
documents = [
    "ChromaDB supports metadata filtering and CRUD operations",
    "Gemini embeddings capture semantic meaning for search applications",
    "HNSW indexing enables efficient approximate nearest neighbor search",
    "Retrieval evaluation requires precision@k and recall@k metrics",
    "Vector databases store embeddings for similarity search"
]
metadata = [
    {"category": "chromadb", "source": "docs"},
    {"category": "embeddings", "source": "research"},
    {"category": "indexing", "source": "paper"},
    {"category": "evaluation", "source": "tutorial"},
    {"category": "vector-db", "source": "textbook"}
]
ids = [f"doc{i+1}" for i in range(len(documents))]

collection.upsert(
    documents=documents,
    metadatas=metadata,
    ids=ids
)
print(f"Added {len(ids)} documents to collection")

Added 5 documents to collection


# Search Pipeline Implementation

* End-to-End Search Workflow:

  * Embeds query using Gemini

  * Queries ChromaDB for similar documents

  * Returns results as pandas DataFrame

* Parameters:

  * query: Search string

  * n_results: Number of results to return

  * filters: Metadata filters (e.g., {"category": "research"})

In [36]:
def search_pipeline(query: str, n_results: int = 3, filters: dict = None) -> pd.DataFrame:
    """End-to-end search pipeline"""
    # Generate query embedding
    query_embedding = gemini_ef([query])[0]

    # Execute search
    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=n_results,
        where=filters
    )

    # Format results
    return pd.DataFrame({
        'Documents': results['documents'][0],
        'IDs': results['ids'][0],
        'Distances': results['distances'][0],
        'Metadata': results['metadatas'][0]
    })

* Demonstrates search functionality

* Queries about search quality metrics

* Returns top 3 relevant documents with:

  * Content snippets

  * Document IDs

  * Similarity scores

  * Metadata



In [37]:
# Example search
print("\nSearch Pipeline Example:")
print(search_pipeline("How to measure search quality?"))


Search Pipeline Example:
                                           Documents   IDs  Distances  \
0  Retrieval evaluation requires precision@k and ...  doc4   0.194412   
1  Gemini embeddings capture semantic meaning for...  doc2   0.261296   
2  Vector databases store embeddings for similari...  doc5   0.308972   

                                           Metadata  
0  {'category': 'evaluation', 'source': 'tutorial'}  
1  {'source': 'research', 'category': 'embeddings'}  
2   {'category': 'vector-db', 'source': 'textbook'}  


#  Retrieval Evaluation Framework

Ground truth data

* Evaluation dataset with:

  * Query texts

  * Ground truth relevant document IDs

* Three sample queries covering:

  * Database features

  * Similarity techniques

  * Evaluation metrics

In [38]:
test_queries = {
    "q1": {
        "text": "ChromaDB features",
        "relevant_ids": ["doc1"]
    },
    "q2": {
        "text": "Similarity measurement techniques",
        "relevant_ids": ["doc5"]
    },
    "q3": {
        "text": "Evaluation metrics for search",
        "relevant_ids": ["doc4"]
    }
}

In [39]:
def evaluate_retrieval(k: int = 3) -> pd.DataFrame:
    """Evaluation framework with metrics calculation"""
    results = []

    for qid, data in tqdm(test_queries.items(), desc="Evaluating queries"):
        # Embed query
        query_embedding = gemini_ef([data["text"]])[0]

        # Retrieve results
        search_results = collection.query(
            query_embeddings=[query_embedding],
            n_results=k
        )
        retrieved_ids = search_results['ids'][0]

        # Calculate metrics
        relevant_set = set(data["relevant_ids"])
        retrieved_set = set(retrieved_ids)
        true_positives = len(relevant_set & retrieved_set)

        precision = true_positives / k
        recall = true_positives / len(relevant_set) if relevant_set else 0

        # Calculate MRR
        reciprocal_rank = 0
        for rank, doc_id in enumerate(retrieved_ids, 1):
            if doc_id in relevant_set:
                reciprocal_rank = 1 / rank
                break

        results.append({
            "query": data["text"],
            "precision@k": precision,
            "recall@k": recall,
            "mrr": reciprocal_rank,
            "retrieved": retrieved_ids
        })

    return pd.DataFrame(results)

# Run Evaluation

* Precision@k: % of retrieved docs that are relevant

* Recall@k: % of relevant docs retrieved

* MRR: Reciprocal rank of first relevant result

* Uses `tqdm` for progress tracking

* Returns dataframe with metrics per query

In [40]:
print("\nRetrieval Evaluation Results:")
eval_results = evaluate_retrieval(k=2)
display(eval_results)


Retrieval Evaluation Results:


Evaluating queries:   0%|          | 0/3 [00:00<?, ?it/s]

,query,precision@k,recall@k,mrr,retrieved
0,ChromaDB features,0.5,1.0,0.5,"[doc5, doc1]"
1,Similarity measurement techniques,0.5,1.0,1.0,"[doc5, doc3]"
2,Evaluation metrics for search,0.5,1.0,1.0,"[doc4, doc2]"


# Calculate averages

In [41]:
avg_metrics = pd.DataFrame({
    "Metric": ["Precision@2", "Recall@2", "MRR"],
    "Average Value": [
        eval_results["precision@k"].mean(),
        eval_results["recall@k"].mean(),
        eval_results["mrr"].mean()
    ]
})
print("\nAverage Metrics:")
display(avg_metrics)


Average Metrics:


,Metric,Average Value
0,Precision@2,0.500000
1,Recall@2,1.000000
2,MRR,0.833333
